# Railway Reservation

**Language:** C

A professional C console project with menu-driven operations and file-based persistence.

**Highlights:** File Handling, Menus, CRUD

This notebook is structured for GitHub and can be opened in Colab later if you want to add cloud execution.


## Source Code

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define DATA_FILE "railway_reservation.dat"

typedef struct {
    int id;
    char name[60];
    char extra1[60];
    char extra2[60];
    float amount;
    int quantity;
    int active;
} Record;

void clearInput(void) {
    int c;
    while ((c = getchar()) != '\n' && c != EOF) {}
}

void trimNewline(char *s) {
    s[strcspn(s, "\n")] = '\0';
}

void readText(const char *prompt, char *buffer, size_t size) {
    printf("%s", prompt);
    fgets(buffer, (int)size, stdin);
    trimNewline(buffer);
}

int nextId(void) {
    FILE *fp = fopen(DATA_FILE, "rb");
    Record r;
    int maxId = 0;
    if (!fp) return 1;
    while (fread(&r, sizeof(r), 1, fp) == 1) {
        if (r.active && r.id > maxId) maxId = r.id;
    }
    fclose(fp);
    return maxId + 1;
}

void saveRecord(Record rec) {
    FILE *fp = fopen(DATA_FILE, "ab");
    if (!fp) {
        perror("Unable to open data file");
        return;
    }
    fwrite(&rec, sizeof(rec), 1, fp);
    fclose(fp);
}

void listRecords(void) {
    FILE *fp = fopen(DATA_FILE, "rb");
    Record r;
    if (!fp) {
        puts("No records found.");
        return;
    }
    puts("\n----- Records -----");
    while (fread(&r, sizeof(r), 1, fp) == 1) {
        if (!r.active) continue;
        printf("ID: %d | Name: %s | Info1: %s | Info2: %s | Amount: %.2f | Qty: %d\n",
               r.id, r.name, r.extra1, r.extra2, r.amount, r.quantity);
    }
    fclose(fp);
}

int updateRecord(int id, const char *field, const char *value, float amount, int quantity) {
    FILE *fp = fopen(DATA_FILE, "rb");
    FILE *tmp = fopen("temp.dat", "wb");
    Record r;
    int updated = 0;
    if (!fp || !tmp) return 0;
    while (fread(&r, sizeof(r), 1, fp) == 1) {
        if (r.id == id && r.active) {
            updated = 1;
            if (strcmp(field, "name") == 0) strncpy(r.name, value, sizeof(r.name)-1);
            else if (strcmp(field, "extra1") == 0) strncpy(r.extra1, value, sizeof(r.extra1)-1);
            else if (strcmp(field, "extra2") == 0) strncpy(r.extra2, value, sizeof(r.extra2)-1);
            else if (strcmp(field, "amount") == 0) r.amount = amount;
            else if (strcmp(field, "quantity") == 0) r.quantity = quantity;
        }
        fwrite(&r, sizeof(r), 1, tmp);
    }
    fclose(fp);
    fclose(tmp);
    remove(DATA_FILE);
    rename("temp.dat", DATA_FILE);
    return updated;
}

int deleteRecord(int id) {
    FILE *fp = fopen(DATA_FILE, "rb");
    FILE *tmp = fopen("temp.dat", "wb");
    Record r;
    int deleted = 0;
    if (!fp || !tmp) return 0;
    while (fread(&r, sizeof(r), 1, fp) == 1) {
        if (r.id == id) {
            deleted = 1;
            continue;
        }
        fwrite(&r, sizeof(r), 1, tmp);
    }
    fclose(fp);
    fclose(tmp);
    remove(DATA_FILE);
    rename("temp.dat", DATA_FILE);
    return deleted;
}

void addSample(void) {
    Record r = {0};
    r.id = nextId();
    r.active = 1;
    printf("Enter name: ");
    fgets(r.name, sizeof(r.name), stdin); trimNewline(r.name);
    readText("Enter info 1: ", r.extra1, sizeof(r.extra1));
    readText("Enter info 2: ", r.extra2, sizeof(r.extra2));
    printf("Enter amount: ");
    scanf("%f", &r.amount);
    printf("Enter quantity: ");
    scanf("%d", &r.quantity);
    clearInput();
    saveRecord(r);
    printf("Saved %s with ID %d\n", "Railway Reservation", r.id);
}

void searchRecord(int id) {
    FILE *fp = fopen(DATA_FILE, "rb");
    Record r;
    int found = 0;
    if (!fp) {
        puts("No data file found.");
        return;
    }
    while (fread(&r, sizeof(r), 1, fp) == 1) {
        if (r.id == id) {
            printf("Found -> ID: %d | Name: %s | Info1: %s | Info2: %s | Amount: %.2f | Qty: %d\n",
                   r.id, r.name, r.extra1, r.extra2, r.amount, r.quantity);
            found = 1;
            break;
        }
    }
    fclose(fp);
    if (!found) puts("Record not found.");
}

int main(void) {
    int choice, id;
    char field[16], value[64];
    float amount;
    int quantity;

    for (;;) {
        printf("\n===== Railway Reservation =====\n");
        puts("1. Add");
        puts("2. List");
        puts("3. Search");
        puts("4. Update");
        puts("5. Delete");
        puts("0. Exit");
        printf("Choose: ");
        if (scanf("%d", &choice) != 1) return 0;
        clearInput();

        if (choice == 0) break;
        switch (choice) {
            case 1:
                addSample();
                break;
            case 2:
                listRecords();
                break;
            case 3:
                printf("Enter ID: ");
                scanf("%d", &id);
                clearInput();
                searchRecord(id);
                break;
            case 4:
                printf("Enter ID: ");
                scanf("%d", &id);
                clearInput();
                readText("Field (name/extra1/extra2/amount/quantity): ", field, sizeof(field));
                if (strcmp(field, "amount") == 0) {
                    printf("New amount: ");
                    scanf("%f", &amount);
                    clearInput();
                    updateRecord(id, field, "", amount, 0);
                } else if (strcmp(field, "quantity") == 0) {
                    printf("New quantity: ");
                    scanf("%d", &quantity);
                    clearInput();
                    updateRecord(id, field, "", 0.0f, quantity);
                } else {
                    readText("New value: ", value, sizeof(value));
                    updateRecord(id, field, value, 0.0f, 0);
                }
                puts("Update completed.");
                break;
            case 5:
                printf("Enter ID: ");
                scanf("%d", &id);
                clearInput();
                puts(deleteRecord(id) ? "Deleted." : "Not found.");
                break;
            default:
                puts("Invalid choice.");
        }
    }
    return 0;
}
